# Results


In [1]:
import sys
from pathlib import Path

USE_KAGGLE = False

if USE_KAGGLE:
    PROJECT_ROOT = Path("/kaggle/input/datasets/gpla77/pro5-code")
    MODELING_ROOT = PROJECT_ROOT / "modeling"

    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    if str(MODELING_ROOT) not in sys.path:
        sys.path.insert(0, str(MODELING_ROOT))

    BEST_MODEL_PATH = Path("/kaggle/input/datasets/gpla77/pro5-model/best_model/best_model.pt")
    NORM_STATS_PATH = PROJECT_ROOT / "data" / "norm_stats.npy"
    OUTPUT_DIR = Path("/kaggle/working/results")
else:
    MODELING_ROOT = Path.cwd()
    PROJECT_ROOT = MODELING_ROOT.parent

    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    if str(MODELING_ROOT) not in sys.path:
        sys.path.insert(0, str(MODELING_ROOT))

    BEST_MODEL_PATH = Path("best_model") / "best_model.pt"
    NORM_STATS_PATH = Path("data") / "norm_stats.npy"
    OUTPUT_DIR = Path("results")
    

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")
print(f"Model: {BEST_MODEL_PATH} (exists: {BEST_MODEL_PATH.exists()})")
print(f"Norm stats: {NORM_STATS_PATH} (exists: {NORM_STATS_PATH.exists()})")
print(f"Output: {OUTPUT_DIR}")

Device: cpu
Model: best_model\best_model.pt (exists: True)
Norm stats: data\norm_stats.npy (exists: True)
Output: results


# Walk

In [2]:
from sample import generate_denormalize_animate_and_save

walk_samples, jump_gif_paths = generate_denormalize_animate_and_save(
    checkpoint_path=str(BEST_MODEL_PATH),
    class_label=0,
    save_dir="results/walk",
    n_samples=12,
    guidance_scale=3.0,
    n_joints=15,
    n_frames=48,
    d_model=384,
    nhead=6,
    num_layers=6,
    num_classes=2,
    dropout=0.1,
    timesteps=1000,
    fps=8,
    norm_stats_path=str(NORM_STATS_PATH),
    device=DEVICE,
    show_plots=False,
    display_gifs=False,
)

d:\STUDIA SZI\Semestr 2\Sztuczna inteligencja w grafice komputerowej\Projekt\repo\AIComputerGraphics\project5-stick-animation\.pro5-sigk\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<div style="display:grid; grid-template-columns:repeat(4, 300px); gap:12px;">
  <img src="results/walk/walk_s1.gif" width="300">
  <img src="results/walk/walk_s2.gif" width="300">
  <img src="results/walk/walk_s3.gif" width="300">
  <img src="results/walk/walk_s4.gif" width="300">
  <img src="results/walk/walk_s5.gif" width="300">
  <img src="results/walk/walk_s6.gif" width="300">
  <img src="results/walk/walk_s7.gif" width="300">
  <img src="results/walk/walk_s8.gif" width="300">
  <img src="results/walk/walk_s9.gif" width="300">
  <img src="results/walk/walk_s10.gif" width="300">
  <img src="results/walk/walk_s11.gif" width="300">
  <img src="results/walk/walk_s12.gif" width="300">
</div>

# Jump

In [3]:
from sample import generate_denormalize_animate_and_save

jump_samples, jump_gif_paths = generate_denormalize_animate_and_save(
    checkpoint_path=str(BEST_MODEL_PATH),
    class_label=1,
    save_dir="results/jump",
    n_samples=12,
    guidance_scale=3.0,
    n_joints=15,
    n_frames=48,
    d_model=384,
    nhead=6,
    num_layers=6,
    num_classes=2,
    dropout=0.1,
    timesteps=1000,
    fps=8,
    norm_stats_path=str(NORM_STATS_PATH),
    device=DEVICE,
    show_plots=False,
    display_gifs=False,
)

<div style="display:grid; grid-template-columns:repeat(4, 300px); gap:12px;">
  <img src="results/jump/jump_s1.gif" width="300">
  <img src="results/jump/jump_s2.gif" width="300">
  <img src="results/jump/jump_s3.gif" width="300">
  <img src="results/jump/jump_s4.gif" width="300">
  <img src="results/jump/jump_s5.gif" width="300">
  <img src="results/jump/jump_s6.gif" width="300">
  <img src="results/jump/jump_s7.gif" width="300">
  <img src="results/jump/jump_s8.gif" width="300">
  <img src="results/jump/jump_s9.gif" width="300">
  <img src="results/jump/jump_s10.gif" width="300">
  <img src="results/jump/jump_s11.gif" width="300">
  <img src="results/jump/jump_s12.gif" width="300">
</div>

In [ ]:
from pathlib import Path
from PIL import Image, ImageSequence


def combine_gifs_into_mosaic(
    gif_paths,
    output_path,
    n_rows,
    n_cols,
    gap=0,
    bg_color=(255, 255, 255),
    duration=125,
    loop=0,
):
    gif_paths = [Path(p) for p in gif_paths][: n_rows * n_cols]
    if not gif_paths:
        raise ValueError("No GIF files provided.")

    gifs = [Image.open(p) for p in gif_paths]
    frame_counts = [getattr(gif, "n_frames", 1) for gif in gifs]
    max_frames = max(frame_counts)

    sizes = []
    for gif in gifs:
        gif.seek(0)
        frame = gif.convert("RGBA")
        sizes.append(frame.size)

    cell_w = max(w for w, h in sizes)
    cell_h = max(h for w, h in sizes)

    canvas_w = n_cols * cell_w + (n_cols - 1) * gap
    canvas_h = n_rows * cell_h + (n_rows - 1) * gap

    output_frames = []

    for frame_idx in range(max_frames):
        canvas = Image.new("RGBA", (canvas_w, canvas_h), bg_color + (255,))

        for idx, gif in enumerate(gifs):
            current_idx = frame_idx % getattr(gif, "n_frames", 1)
            gif.seek(current_idx)
            frame = gif.convert("RGBA")

            row = idx // n_cols
            col = idx % n_cols
            x = col * (cell_w + gap)
            y = row * (cell_h + gap)

            canvas.paste(frame, (x, y), frame)

        output_frames.append(canvas.convert("P", palette=Image.Palette.ADAPTIVE))

    output_frames[0].save(
        output_path,
        save_all=True,
        append_images=output_frames[1:],
        duration=duration,
        loop=loop,
        optimize=True,
    )

In [26]:
from pathlib import Path

gif_paths = sorted(Path("results/jump").glob("*.gif"))

combine_gifs_into_mosaic(
    gif_paths=gif_paths,
    output_path="results/jump_grid.gif",
    n_rows=3,
    n_cols=4,
    gap=0,
    duration=125,
    loop=0,
)

In [27]:
from pathlib import Path

gif_paths = sorted(Path("results/walk").glob("*.gif"))

combine_gifs_into_mosaic(
    gif_paths=gif_paths,
    output_path="results/walk_grid.gif",
    n_rows=3,
    n_cols=4,
    gap=0,
    duration=125,
    loop=0,
)